# 01 — Introduction to Projective Geometric Algebra (PGA)

This notebook introduces **Projective Geometric Algebra (PGA)** — an extension of GA that unifies points, lines, and planes in a single framework. PGA is particularly powerful for robotics because it naturally represents rigid body motions.

## Learning Objectives

- Understand why PGA is useful for geometry and robotics
- Explain the role of the null basis e0 (homogenization)
- Compare VGA and PGA signatures
- Construct PGA2d and PGA3d algebras in AMSA

In [ ]:
# Setup
import matplotlib.pyplot as plt
import numpy as np

from amsa import Algebra

alg_2d = Algebra.pga2d()
alg_3d = Algebra.pga3d()

## 1.1 The Motivation for PGA

In VGA, we can represent points and vectors, but:

- **Lines** require bivectors (in 2D) or more complex constructions
- **Translations** are not naturally represented — they require limits of rotations
- **Infinity** is problematic — parallel lines don't meet

PGA solves this by adding a **homogeneous dimension** (the null basis e0). This unifies:

| VGA | PGA | What it represents |
|-----|-----|-------------------|
 | Vector | Line (using `e1`, `e2`, `e0`) | Direction + position |
 | — | Point (using `e01`, `e02`, `e12`) | Homogeneous coordinates |
 | — | Motor | Translation + rotation |

As Charles Gunn (SIGGRAPH 2019) puts it: *PGA provides a coordinate-free framework for doing Euclidean geometry.*

## 1.2 PGA Signatures

PGA uses a degenerate metric — one dimension is nilpotent (squares to zero):

- **PGA2d**: signature (0, 1, 1) — one null + two Euclidean dimensions
- **PGA3d**: signature (0, 1, 1, 1) — one null + three Euclidean dimensions

The null basis `e0` satisfies $e_0^2 = 0$, which enables translations.

In [ ]:
print("=== PGA2d ===")
print("Dimension:", alg_2d.spec.dimension)
print("Signature:", alg_2d.spec.signature)
print("Blade count:", alg_2d.spec.blade_count)
print("\n=== PGA3d ===")
print("Dimension:", alg_3d.spec.dimension)
print("Signature:", alg_3d.spec.signature)
print("Blade count:", alg_3d.spec.blade_count)

## 1.3 PGA2d Blade Overview

PGA2d has 8 blades: one for each subset of the 3 basis directions, so the count is still $2^3 = 8$. Let's examine them.

In [ ]:
print("PGA2d blades:")
grades = alg_2d.spec.grades_of_blades()
for blade_index in range(alg_2d.spec.blade_count):
    name = alg_2d.spec.blade_name(blade_index)
    grade = grades[blade_index]
    print(f"  Blade {blade_index}: '{name}' (grade {grade})")

## 1.4 Understanding the Null Basis e0

The key insight: in PGA, we add a dimension with signature 0 (null):

- $e_0 \cdot e_0 = 0$ (nilpotent)
- This enables the homogenization of Euclidean geometry

The current AMSA codebase uses two complementary PGA2d views:

- a **vector-form point** used naturally by line-line meet algebra, written as $P = w e_0 - y e_1 + x e_2$
- a **bivector-form point** used by plotting and motor examples, written as $P^* = x e_{01} + y e_{02} + w e_{12}$

These are related by `poincare_dual()` in the current AMSA implementation.


In [ ]:
# Vector-form point used by meet/join algebra
point = alg_2d.multivector({"e0": 1.0, "e1": -2.0, "e2": 1.0})
point_plot = point.poincare_dual()

print("Vector-form point:", point.values)
print("Plotting/motor form via poincare_dual():", point_plot.values)
print(
    "Coordinates: x =",
    point.component("e2") / point.component("e0"),
    ", y =",
    -point.component("e1") / point.component("e0"),
)

# At infinity (parallel lines meet here)
point_inf = alg_2d.multivector({"e0": 0.0, "e1": 0.0, "e2": 1.0})  # e0 = 0 -> ideal point
print("\nIdeal point:", point_inf.values)
print("Ideal point in plotting form:", point_inf.poincare_dual().values)
print("  (direction only, no position)")


### Lines and incidence

Lines in the current AMSA PGA2d examples are usually written in bivector form
`L = a e01 + b e02 + c e12`, which corresponds to the Euclidean line equation
`a x + b y + c = 0`.

For this notebook we will verify incidence by extracting `(x, y)` from a vector-form
point and plugging those coordinates into the line equation. That keeps the example
aligned with the current library conventions without mixing representations.


In [ ]:
# Define a line: x + y + 1 = 0 (passes through points where x+y=-1)
line = alg_2d.multivector({
    "e01": 1.0,
    "e02": 1.0,
    "e12": 1.0,
})

print("Line (e01 + e02 + e12):", line.values)

# Check incidence by plugging extracted coordinates into a x + b y + c = 0
p1 = alg_2d.multivector({"e0": 1.0, "e1": 1.0, "e2": 0.0})   # (0, -1)
p2 = alg_2d.multivector({"e0": 1.0, "e1": 2.0, "e2": 1.0})   # (1, -2)

def point_xy(point):
    return (
        point.component("e2") / point.component("e0"),
        -point.component("e1") / point.component("e0"),
    )

def line_residual(point, line):
    x, y = point_xy(point)
    return (
        line.component("e01") * x
        + line.component("e02") * y
        + line.component("e12")
    )

print("\nPoint (0, -1) residual:", line_residual(p1, line))
print("Point (1, -2) residual:", line_residual(p2, line))


## 1.6 Visualizing PGA2d

Let's visualize points and lines in PGA2d.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Draw coordinate axes
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)

# Points
points = [
    alg_2d.multivector({"e01": 1.0, "e02": 2.0, "e12": 1.0}),
    alg_2d.multivector({"e01": -1.0, "e02": 1.0, "e12": 1.0}),
    alg_2d.multivector({"e01": 2.0, "e02": -1.0, "e12": 1.0})
]

for i, p in enumerate(points):
    x = p.component("e01") / p.component("e12")
    y = p.component("e02") / p.component("e12")
    ax.scatter(x, y, s=100, zorder=5)
    ax.text(x + 0.1, y + 0.1, f'P{i+1}', fontsize=10)

# Line: x + y + 1 = 0
x_vals = np.linspace(-3, 2, 100)
y_vals = -x_vals - 1
ax.plot(x_vals, y_vals, 'b-', linewidth=2, label='Line: x + y + 1 = 0')

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.set_title('PGA2d: Points and Lines', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 1.7 Why PGA Matters for Robotics

PGA provides a unified framework for:

1. **Rigid body motions**: Motors (translations + rotations)
2. **Lines and points**: Natural intersection and join operations
3. **Projective duality**: Points ↔ Lines, planes ↔ points
4. **Kinematics**: Single algebra for forward and inverse kinematics
5. **Computer vision**: Homogeneous coordinates are standard

As noted in the original PGA paper (Gunn, arXiv:1901.05873):
> *PGA solves the dual problems of kinematics and geometry in a unified, coordinate-free way.*

## 1.8 Summary

We covered:

- **PGA motivation**: Unifies points, lines, translations in one algebra
- **Null basis e0**: Enables homogenization, $e_0^2 = 0$
- **PGA2d**: Signature `(0, 1, 1)`, 8 blades
- **PGA3d**: Signature `(0, 1, 1, 1)`, 16 blades
- **Vector-form points**: `w e0 - y e1 + x e2` arise naturally from meet/join algebra
- **Bivector-form points**: `x e01 + y e02 + w e12` are what `amsa.viz` plots
- **Lines**: In the current geometry examples, `a e01 + b e02 + c e12` corresponds to `a x + b y + c = 0`
- **Incidence**: With AMSA's current point/line pairing, vector-form points satisfy `P | L = 0`
- **Robotics**: Motors for rigid body motion

In the next notebook, we'll explore lines, points, and their meet/join operations.


## Exercises

### ⭐ Easy

**1.1** Create a vector-form point at (3, 4) in PGA2d and verify its homogeneous coordinate `e0` is 1. Then apply `poincare_dual()` and inspect the plotting form.

In [ ]:
# Your turn: ⭐ Exercise 1.1
point = alg_2d.multivector({"e01": 3.0, "e02": 4.0, "e12": 1.0})
# TODO: Verify components
raise NotImplementedError("Implement exercise 1.1")

### ⭐⭐ Medium

**1.2** Create two parallel lines in PGA2d: L1: x = 0 and L2: x = 2. Compute their meet (intersection) — what happens? What does this tell you about PGA and parallel lines?

In [ ]:
# Your turn: ⭐⭐ Exercise 1.2
# L1: x = 0 → e1 component with e0 = 0
# L2: x = 2 → 
# TODO: Compute meet of parallel lines
raise NotImplementedError("Implement exercise 1.2")

### ⭐⭐⭐ Challenge

**1.3** Write a function that converts a classical line `a x + b y + c = 0` into the PGA2d bivector `a e01 + b e02 + c e12`. Verify it works for at least 3 different lines.

In [ ]:
# Your turn: ⭐⭐⭐ Exercise 1.3
def classical_line_to_pga(a, b, c):
    """Convert a x + b y + c = 0 to a PGA2d line bivector."""
    # TODO: Implement
raise NotImplementedError("Implement classical_line_to_pga function")

# Test cases
test_lines = [(1, 1, 1), (2, -1, 0), (0, 1, -3)]
for a, b, c in test_lines:
    line = classical_line_to_pga(a, b, c)
    print(f"Line {a}x + {b}y + {c} = 0:", line.values)

## Attribution

This notebook draws on:

- **Projective Geometric Algebra** — Charles G. Gunn
  https://arxiv.org/abs/1901.05873
- **SIGGRAPH 2019 Course Notes** — Charles G. Gunn
  https://arxiv.org/abs/2002.04509
- **PGABLE Tutorial** — Zachary Leger and Stephen Mann
  https://cs.uwaterloo.ca/~smann/PGABLE/PGAtutorial.pdf
- **Geometric Algebra for Computer Graphics** — John Vince
  https://link.springer.com/book/10.1007/978-1-84628-997-2